---
# 04. 회귀분석 — FPI 영향 분석

## 분석 개요

FPI가 독립 브랜드의 별점과 폐업여부에 미치는 영향을 검증한다.

| 단계 | 내용 |
|---|---|
| STEP 4-1 | 데이터 준비 및 통제변수 선정 |
| STEP 4-2 | OLS + HC3 (FPI → 별점) |
| STEP 4-3 | Logistic + HC3 (FPI → 폐업여부) |

**분석 설계**

| 모델 | 종속변수 | 독립변수 | 통제변수 |
|---|---|---|---|
| OLS + HC3 | 별점 | fpi_300m | log(review_count), neighborhood 더미 |
| Logistic + HC3 | 폐업여부 (is_open) | fpi_300m | stars, log(review_count), neighborhood 더미 |

**핵심 연구질문**
- Q1. FPI가 높을수록 독립 브랜드 별점이 낮아지는가?
- Q2. 별점·리뷰수·상권을 통제해도 FPI가 높을수록 폐업 확률이 높아지는가?

**입력 데이터**
- `biz_indie_with_groups_burger.csv`

## 공통 라이브러리 및 설정

In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore')

fm._load_fontmanager(try_read_cache=False)
plt.rc('font', family='Malgun Gothic')
plt.rc('axes', unicode_minus=False)

PATH_to_data = "../../results"
PATH_to_save = "../../results"

# 데이터 로드
indie_df = pd.read_csv(f"{PATH_to_data}/biz_indie_with_groups_burger.csv")

print(f"독립 브랜드: {len(indie_df):,}개")
print(f"\n영업 여부:")
print(indie_df['is_open'].value_counts())
print(f"\nFPI 구간:")
print(indie_df['fpi_group'].value_counts())

독립 브랜드: 775개

영업 여부:
is_open
1    535
0    240
Name: count, dtype: int64

FPI 구간:
fpi_group
HP    296
LP    296
NP    183
Name: count, dtype: int64


---
## STEP 4-1. 데이터 준비 및 통제변수 선정

**통제변수 선정 기준 (restaurant과 동일)**
1. 종속변수에 영향을 미치는 변수
2. 독립변수(FPI)와 상관관계가 있는 변수

**로지스틱 회귀 추가 통제변수: stars**

폐업여부는 별점의 영향을 강하게 받는다.
별점을 통제해야 FPI의 순수한 폐업 영향을 추출할 수 있다.

In [3]:
# 전처리
indie_df['log_review'] = np.log(indie_df['review_count'] + 1)
indie_df['neighborhood'] = indie_df['neighborhood'].fillna('Unknown')

# 상관관계 확인
print("=== 후보 변수 × 종속변수 상관계수 ===")
candidates = ['log_review', 'fpi_300m', 'stars']
targets = ['stars', 'is_open']
corr_df = indie_df[candidates + ['is_open']].corr()
print(corr_df[['stars', 'is_open']].loc[candidates].round(4))

print("\n=== neighborhood별 평균 별점 및 FPI ===")
nb_stats = indie_df.groupby('neighborhood').agg(
    평균별점=('stars', 'mean'),
    평균FPI=('fpi_300m', 'mean'),
    업체수=('business_id', 'count')
).sort_values('업체수', ascending=False).head(10)
print(nb_stats.round(2))

# neighborhood 더미 변수
nb_dummies = pd.get_dummies(indie_df['neighborhood'], prefix='nb', drop_first=True)
indie_df = pd.concat([indie_df.reset_index(drop=True),
                      nb_dummies.reset_index(drop=True)], axis=1)
dummy_cols = [c for c in indie_df.columns if c.startswith('nb_')]

print(f"\nneighborhood 더미 변수: {len(dummy_cols)}개")
print(f"최종 분석 데이터: {len(indie_df):,}개")

=== 후보 변수 × 종속변수 상관계수 ===
             stars  is_open
log_review  0.1527   0.2753
fpi_300m   -0.0876  -0.0435
stars       1.0000  -0.0458

=== neighborhood별 평균 별점 및 FPI ===
               평균별점  평균FPI  업체수
neighborhood                   
The Strip      3.37   3.34  138
Southeast      3.30   1.68   96
Westside       3.72   1.43   87
Unknown        3.69   1.19   78
Spring Valley  3.62   2.65   77
Eastside       3.57   1.48   68
Downtown       3.79   0.86   66
Southwest      3.29   0.87   33
Northwest      3.52   1.45   28
Centennial     3.60   1.95   25

neighborhood 더미 변수: 16개
최종 분석 데이터: 775개


---
## STEP 4-2. OLS + HC3 (FPI → 별점)

restaurant과 동일한 방법론으로 FPI가 별점에 미치는 순수한 영향을 검증한다.
HC3 Robust Standard Error로 이분산 보정.

In [4]:
X_stars = indie_df[['fpi_300m', 'log_review'] + dummy_cols].astype(float)
X_stars = sm.add_constant(X_stars)
y_stars = indie_df['stars']

model_stars = sm.OLS(y_stars, X_stars).fit(cov_type='HC3')

print("=" * 60)
print("모델 1 — FPI → 별점 (OLS + HC3)")
print("=" * 60)
print(f"\n[FPI 계수]")
print(f"  회귀계수 : {model_stars.params['fpi_300m']:.4f}")
print(f"  P-value  : {model_stars.pvalues['fpi_300m']:.4f}")
print(f"  95% CI   : [{model_stars.conf_int().loc['fpi_300m', 0]:.4f}, "
      f"{model_stars.conf_int().loc['fpi_300m', 1]:.4f}]")
print(f"  유의미   : {'✅ p<0.05' if model_stars.pvalues['fpi_300m'] < 0.05 else '❌ p≥0.05'}")
print(f"\n[모델 전체]")
print(f"  R²       : {model_stars.rsquared:.4f}")
print(f"  Adj R²   : {model_stars.rsquared_adj:.4f}")
print(f"  F-stat   : {model_stars.fvalue:.4f} (p={model_stars.f_pvalue:.4f})")
print(f"  관측치   : {int(model_stars.nobs):,}개")

모델 1 — FPI → 별점 (OLS + HC3)

[FPI 계수]
  회귀계수 : -0.0271
  P-value  : 0.0479
  95% CI   : [-0.0539, -0.0002]
  유의미   : ✅ p<0.05

[모델 전체]
  R²       : 0.0825
  Adj R²   : 0.0607
  F-stat   : 3.4997 (p=0.0000)
  관측치   : 775개


---
## STEP 4-3. Logistic + HC3 (FPI → 폐업여부)

**연구질문**: 별점·리뷰수·상권을 통제한 후에도
FPI가 높을수록 독립 브랜드의 폐업 확률이 높아지는가?

**해석 방법**
- 종속변수: is_open (1=영업 중, 0=폐업)
- 회귀계수 음수: FPI 높을수록 영업 중 확률 낮아짐
- Odds Ratio < 1: FPI 1 증가 시 영업 중 확률이 OR배로 감소
- 단, 본 분석에서 FPI는 유의미하지 않아(p=0.401) 해석에 주의가 필요하다

In [9]:
X_open = indie_df[['fpi_300m', 'stars', 'log_review'] + dummy_cols].astype(float)
X_open = sm.add_constant(X_open)
y_open = indie_df['is_open']

model_open = sm.Logit(y_open, X_open).fit(cov_type='HC3', disp=0)

print("=" * 60)
print("모델 2 — FPI → 폐업여부 (Logistic + HC3)")
print("=" * 60)
print(f"\n[FPI 계수]")
print(f"  회귀계수  : {model_open.params['fpi_300m']:.4f}")
print(f"  P-value   : {model_open.pvalues['fpi_300m']:.4f}")
print(f"  Odds Ratio: {np.exp(model_open.params['fpi_300m']):.4f}")
print(f"  95% CI    : [{model_open.conf_int().loc['fpi_300m', 0]:.4f}, "
      f"{model_open.conf_int().loc['fpi_300m', 1]:.4f}]")
print(f"  유의미    : {'✅ p<0.05' if model_open.pvalues['fpi_300m'] < 0.05 else '❌ p≥0.05'}")
print(f"\n[모델 전체]")
print(f"  Log-Likelihood : {model_open.llf:.4f}")
print(f"  Pseudo R²      : {model_open.prsquared:.4f}")
print(f"  관측치         : {int(model_open.nobs):,}개")

# 통제변수 영향 확인
print(f"\n[주요 통제변수 계수]")
for var in ['stars', 'log_review']:
    print(f"  {var:15s}: coef={model_open.params[var]:.4f}, "
          f"p={model_open.pvalues[var]:.4f}, "
          f"OR={np.exp(model_open.params[var]):.4f}")

모델 2 — FPI → 폐업여부 (Logistic + HC3)

[FPI 계수]
  회귀계수  : -0.0342
  P-value   : 0.4008
  Odds Ratio: 0.9664
  95% CI    : [-0.1140, 0.0456]
  유의미    : ❌ p≥0.05

[모델 전체]
  Log-Likelihood : -439.1886
  Pseudo R²      : 0.0843
  관측치         : 775개

[주요 통제변수 계수]
  stars          : coef=-0.2997, p=0.0046, OR=0.7411
  log_review     : coef=0.4828, p=0.0000, OR=1.6206


In [10]:
print("=== 영업여부별 평균 별점 ===")
print(indie_df.groupby('is_open')['stars'].describe().round(2))

print("\n=== 별점 구간별 폐업률 ===")
indie_df['stars_group'] = pd.cut(indie_df['stars'],
                                  bins=[0, 2.5, 3.0, 3.5, 4.0, 5.0],
                                  labels=['~2.5', '3.0', '3.5', '4.0', '4.5~5.0'])
print(indie_df.groupby('stars_group')['is_open'].mean().round(2))

=== 영업여부별 평균 별점 ===
         count  mean   std  min  25%  50%  75%  max
is_open                                            
0        240.0  3.60  0.73  1.0  3.0  3.5  4.0  5.0
1        535.0  3.52  0.79  1.0  3.0  3.5  4.0  5.0

=== 별점 구간별 폐업률 ===
stars_group
~2.5       0.79
3.0        0.64
3.5        0.65
4.0        0.72
4.5~5.0    0.67
Name: is_open, dtype: float64


---
### 모델 2 해석 주의사항

**별점 계수 방향 문제**

로지스틱 회귀 결과에서 별점 계수가 음수(coef=-0.300, OR=0.741)로 나타났다.
종속변수가 is_open(1=영업 중)이므로 이는 별점↑ → 영업 중 확률↓ 즉, 별점이 높을수록
폐업 가능성이 높다는 의미인데, 이는 직관과 반대되는 결과다.

실제 데이터를 확인한 결과:
- 영업 중 브랜드 평균 별점: 3.52
- 폐업 브랜드 평균 별점: 3.60

데이터 자체에서 폐업 브랜드의 평균 별점이 더 높게 나타났다.

**원인: 횡단면 데이터의 한계**

is_open은 데이터 수집 시점(2017년) 기준이다.
별점이 높았던 브랜드도 이후 시점에 폐업했을 가능성이 있고,
별점이 낮아도 현재까지 영업 중인 브랜드가 존재한다.
즉, 현재 시점의 별점이 폐업의 선행 지표로 작동하지 않는 구조다.

**결론**

별점 → 폐업 경로가 데이터상 성립하지 않으므로
매개효과 분석(FPI → 별점 → 폐업)은 통계적으로 검증되지 않았다.

FPI가 폐업에 직접적인 영향을 미치지 않는다는 결과(p=0.401)와
리뷰수가 영업 지속의 핵심 변수라는 결과(OR=1.621, p=0.000)는 유효하다.

> 향후 연구에서는 패널 데이터 또는 생존 분석(Survival Analysis)을 적용하여
> 시간의 흐름에 따른 폐업 예측 모델을 구축하는 것이 필요하다.

In [12]:
# FPI → log(리뷰수) 회귀
X_a = indie_df[['fpi_300m'] + dummy_cols].astype(float)
X_a = sm.add_constant(X_a)
model_a_review = sm.OLS(np.log(indie_df['review_count'] + 1), X_a).fit(cov_type='HC3')

print(f"경로 a: FPI → log(리뷰수)")
print(f"  coef: {model_a_review.params['fpi_300m']:.4f}")
print(f"  p-value: {model_a_review.pvalues['fpi_300m']:.4f}")

경로 a: FPI → log(리뷰수)
  coef: 0.0215
  p-value: 0.4126


---
## STEP 4 결과 정리

### 분석 데이터 개요

| 항목 | 내용 |
|---|---|
| 분석 대상 | 독립 브랜드 775개 |
| 영업 중 | 535개 (69.0%) |
| 폐업 | 240개 (31.0%) |
| FPI 구간 | NP 183개 / LP 296개 / HP 296개 |
| 통제변수 | log(review_count), neighborhood 더미 16개 |

**통제변수 선정 근거**

| 변수 | 별점 상관 | 영업여부 상관 | 채택 |
|---|---|---|---|
| log(review_count) | 0.153 | 0.275 | ✅ |
| fpi_300m | -0.088 | -0.044 | 독립변수 |
| neighborhood 더미 | FPI 범위 0.86~3.34 | - | ✅ |

---

### 모델 1 — FPI → 별점 (OLS + HC3)

| 항목 | 값 |
|---|---|
| FPI 회귀계수 | -0.0271 |
| p-value | 0.048 ✅ |
| 95% CI | [-0.054, -0.000] |
| R² | 0.0825 |
| Adj R² | 0.0607 |
| 관측치 | 775개 |

FPI가 1 증가할 때 독립 브랜드 별점이 평균 0.027점 감소한다.
전체 Restaurants(-0.012) 대비 약 2.3배 강한 효과로,
업종을 좁히면 FPI 효과가 더 선명하게 드러남을 확인하였다.

---

### 모델 2 — FPI → 폐업여부 (Logistic + HC3)

종속변수: is_open (1=영업 중, 0=폐업)

| 항목 | 값 |
|---|---|
| FPI 회귀계수 | -0.0342 |
| p-value | 0.401 ❌ |
| Odds Ratio | 0.966 |
| Pseudo R² | 0.0843 |
| 관측치 | 775개 |

**주요 통제변수**

| 변수 | 계수 | Odds Ratio | p-value |
|---|---|---|---|
| 별점 | -0.2997 | 0.741 | 0.005 |
| log(리뷰수) | +0.4828 | 1.621 | 0.000 ✅ |

FPI는 폐업여부에 직접적인 영향을 미치지 않는다(p=0.401).
리뷰수가 많을수록 영업 지속 가능성이 높아지며(OR=1.621),
이는 소비자 참여도가 브랜드 생존의 핵심 변수임을 시사한다.

---

### 매개효과 분석 시도 및 한계

FPI → 별점 → 폐업, FPI → 리뷰수 → 폐업 두 가지 매개 경로를 검토하였으나
모두 성립하지 않았다.

| 매개 경로 | 경로 a | 경로 b | 성립 여부 |
|---|---|---|---|
| FPI → 별점 → 폐업 | p=0.048 ✅ | 방향 불일치 ❌ | ❌ |
| FPI → 리뷰수 → 폐업 | p=0.413 ❌ | p=0.000 ✅ | ❌ |

**별점-폐업 방향 불일치 원인**

데이터 확인 결과 영업 중 브랜드 평균 별점(3.52)이
폐업 브랜드 평균 별점(3.60)보다 오히려 낮게 나타났다.

이는 횡단면 데이터의 구조적 한계에서 비롯된다.
is_open은 데이터 수집 시점(2017년) 기준이므로,
별점이 높았던 브랜드가 이후 시점에 폐업했거나
별점이 낮아도 현재까지 영업 중인 브랜드가 혼재한다.
즉 현재 시점의 별점이 폐업의 선행 지표로 작동하지 않는 구조다.

> **분석 한계 명시**
>
> 폐업 분석은 횡단면 데이터의 한계로 인과관계 검증에 제약이 있다.
> 향후 연구에서는 패널 데이터 또는 생존 분석(Survival Analysis)을 적용하여
> 시간의 흐름에 따른 폐업 예측 모델을 구축하는 것이 필요하다.

---

### 종합

| 분석 | 결과 | 유의미 |
|---|---|---|
| FPI → 별점 | coef=-0.027 (팀플 대비 2.3배) | ✅ |
| FPI → 폐업 직접 효과 | p=0.401 | ❌ |
| 폐업 주요 결정 변수 | 리뷰수 (OR=1.621) | ✅ |
| 매개효과 | 두 경로 모두 성립하지 않음 | ❌ |

FPI는 독립 브랜드의 별점에 유의미한 부정적 영향을 미치나,
폐업에 대한 직접·간접 경로는 횡단면 데이터 한계로 검증되지 않았다.
이후 텍스트 분석(STEP 5)에서 고압력 환경에서의 생존 전략을 탐색한다.